### RAG pipeling - Data ingestion to vector DB pipeline

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [8]:
### Read all the pdf's inside the directory

def load_pdfs_with_metadata(directory_path):
    all_docs = []

    for file in os.listdir(directory_path):
        if file.endswith(".pdf"):
            file_path = os.path.join(directory_path, file)

            # Load PDF
            loader = PyPDFLoader(file_path)
            documents = loader.load()

            # Add metadata
            for doc in documents:
                doc.metadata.update({
                    "source_file": file,
                    "file_path": file_path,
                    "file_type": "pdf"
                })

            all_docs.extend(documents)

    return all_docs

In [9]:
all_pdf_documents=load_pdfs_with_metadata("./data/pdf")

In [10]:
print(all_pdf_documents)

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-10T10:40:32+00:00', 'author': '', 'keywords': '', 'moddate': '2026-04-10T10:40:32+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/pdf/sanskar_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sanskar_resume.pdf', 'file_path': './data/pdf/sanskar_resume.pdf', 'file_type': 'pdf'}, page_content='Sanskar Vishwakarma\n+91 93215 97049 — sanskarv2004@gmail.com — Linkedin — Github\nEducation\nBachelor of Engineering (Computer Engineering)Jun 2021 – Jun 2025\nThakur College of Engineering and Technology9.48 CGPA\nHigher Secondary Certificate (HSC - PCM)Jun 2021\nNirmala College of Science and Commerce94%\nTechnical Skills\nCore Concepts:Data Structures, OOP, DBMS, API Integration.\nProgramming Languages:Java, JavaScript, TypeSc

In [11]:
### Chunking the documents into text

def chunk_documents(documents, chunk_size=1000, chunk_overlap=200):
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    
    chunked_docs = splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(chunked_docs)} chunks")

    if chunked_docs:
        print(f"\nExample chunk")
        print(f"\nContent: {chunked_docs[0].page_content[:200]}")
        print(f"\nMetadata: {chunked_docs[0].metadata}")

    return chunked_docs

In [12]:
chunks=chunk_documents(all_pdf_documents)

Split 1 documents into 4 chunks

Example chunk

Content: Sanskar Vishwakarma
+91 93215 97049 — sanskarv2004@gmail.com — Linkedin — Github
Education
Bachelor of Engineering (Computer Engineering)Jun 2021 – Jun 2025
Thakur College of Engineering and Technolog

Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-10T10:40:32+00:00', 'author': '', 'keywords': '', 'moddate': '2026-04-10T10:40:32+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/pdf/sanskar_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sanskar_resume.pdf', 'file_path': './data/pdf/sanskar_resume.pdf', 'file_type': 'pdf'}


### Embedding and VectorDB

In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name="all-MiniLM-L6-v2"):
        """
        Intialise the embedding manager

        Args:
            model_name: HugginFace model for sentence embeddings
        """

        self.model_name=model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimesion: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loadig model {self.model_name}:{e}")
            raise

    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """
        generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts),embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        
        return embeddings


### initialise the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7783.74it/s]


Model loaded successfully. Embedding dimesion: 384


### Vector Store

In [15]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector Store"""

    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="./data/vector_store"):
        """
        Initialises the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialise_store()

    def _initialise_store(self):
        """Intialise ChromaDB client and collection"""
        try:
            # Creating ChromaDB persistent client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector Store initialised, Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initialising vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
            """
            Add documents and their embeddings to the vector store
            
            Args:
                documents: List of LangChain documents
                embeddings: Corresponding embeddings for the documents
            """
            if len(documents) != len(embeddings):
                raise ValueError("Number of documents must match number of embeddings")
            
            print(f"Adding {len(documents)} documents to vector store...")
            
            # Prepare data for ChromaDB
            ids = []
            metadatas = []
            documents_text = []
            embeddings_list = []
            
            for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
                # Generate unique ID
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)
                
                # Prepare metadata
                metadata = dict(doc.metadata)
                metadata['doc_index'] = i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)
                
                # Document content
                documents_text.append(doc.page_content)
                
                # Embedding
                embeddings_list.append(embedding.tolist())
            
            # Add to collection
            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )
                print(f"Successfully added {len(documents)} documents to vector store")
                print(f"Total documents in collection: {self.collection.count()}")
                
            except Exception as e:
                print(f"Error adding documents to vector store: {e}")
                raise


vectorStore=VectorStore()

vectorStore
        

Vector Store initialised, Collection: pdf_documents
Existing documents in collection: 0


In [18]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-10T10:40:32+00:00', 'author': '', 'keywords': '', 'moddate': '2026-04-10T10:40:32+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/pdf/sanskar_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sanskar_resume.pdf', 'file_path': './data/pdf/sanskar_resume.pdf', 'file_type': 'pdf'}, page_content='Sanskar Vishwakarma\n+91 93215 97049 — sanskarv2004@gmail.com — Linkedin — Github\nEducation\nBachelor of Engineering (Computer Engineering)Jun 2021 – Jun 2025\nThakur College of Engineering and Technology9.48 CGPA\nHigher Secondary Certificate (HSC - PCM)Jun 2021\nNirmala College of Science and Commerce94%\nTechnical Skills\nCore Concepts:Data Structures, OOP, DBMS, API Integration.\nProgramming Languages:Java, JavaScript, TypeSc

In [22]:
### convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts

### Generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)
embeddings

### Store the embeddings and chunks into vector store
vectorStore.add_documents(chunks,embeddings)



generating embeddings for 4 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.24it/s]

Generated embeddings with shape: (4, 384)
Adding 4 documents to vector store...
Successfully added 4 documents to vector store
Total documents in collection: 4
